# Hierarchical Stackelberg Security Problem

## Phase 0–1: Authoritative Architecture and Governing Contract

**Status:** architecture specification only. This notebook intentionally contains no implementation or executable cells.

This notebook is the authoritative design contract for a hierarchical Stackelberg Security Problem with one Defender (leader) and one Attacker (follower). Future phases shall extend this architecture without changing its hierarchy, public interfaces, solver responsibilities, or artifact contracts unless explicitly instructed.

## 1. Problem definition

### Defender (leader)

- Continuous decision variable: sensor position \(z_{sensor}\).
- Objective: maximize \(J_D = w_{pod} PoD_{Normalized} + w_{cover} CoverageArea_{Normalized}\).
- Evaluate only the refined **Best-found Attacker Response**, never a coarse Bellman result.

### Attacker (follower)

- Continuous decisions whenever possible: switching point, glide trajectory, speed, and flight-path angle \(\gamma\).
- Objective: minimize \(J_A = w_{pod} PoD_{Normalized} + w_t Time_{Normalized}\).
- Bellman and NLP shall use exactly the same objective and weights. Weights shall never be altered to manufacture candidate diversity.

## 2. Immutable optimization hierarchy

Every future implementation shall preserve this order:

1. Continuous Defender optimization
2. Environment construction
3. CasADi symbolic detection model
4. 4D stage-cost construction
5. 2D projected cost — visualization only
6. Multi-start coarse Bellman
7. Bellman candidate filtering
8. Bellman warm-start construction
9. Multi-start CasADi NLP
10. Best-found Attacker Response selection
11. Defender objective evaluation
12. Continuation of continuous Defender optimization
13. Final Stackelberg solution

The 2D projection is never an optimization state or a substitute for the 4D cost. The nesting relation is always **Bellman → NLP**, inside the continuous Defender optimization.

## 3. Solver responsibilities

### Multi-start coarse Bellman

**Single responsibility:** generate diverse coarse topologies and valid warm-start candidates under the true Attacker objective. It may produce coarse switching points, state/control paths, objective components, feasibility diagnostics, and provenance. It shall not claim a final solution, alter weights, evaluate the Defender, or replace NLP refinement.

### Bellman candidate filtering

**Single responsibility:** validate, deduplicate, rank, and select Bellman candidates without changing their objective. Filtering criteria and rejection reasons must be exported.

### Multi-start CasADi NLP

**Single responsibility:** refine switching point, trajectory, controls, and objective from each selected Bellman warm start. Every start retains solver status, feasibility, objective components, and source candidate ID.

### Best-found Attacker Response

The best feasible refined NLP result under \(J_A\) is called the **Best-found Attacker Response**. It shall not be labeled a global optimum.

## 4. Module architecture and public interfaces

| Module | Required input | Required output | Prohibited responsibility |
|---|---|---|---|
| Configuration | versioned scenario and solver configuration | validated immutable configuration | geometry, solving, plotting |
| Geometry | terrain and spatial parameters | terrain, LOS, bounds, masks, coverage primitives | optimization and plotting |
| Environment | configuration plus geometry artifacts | immutable environment instance | solving and plotting |
| Detection | environment plus states and \(z_{sensor}\) | per-sensor and fused detection quantities | optimization and plotting |
| Dynamics | state, control, time step, vehicle parameters | next state and transition diagnostics | detection, optimization, plotting |
| Objectives | normalized components and fixed weights | \(J_A\), \(J_D\), component breakdowns | solver logic and plotting |
| 4D Stage Cost | 4D state/control grid, dynamics, detection, objective | versioned 4D local-cost artifact | projection, solving, plotting |
| 2D Projection | exported 4D cost artifact | exported 2D visualization artifact | optimization or policy decisions |
| Bellman | environment, 4D cost, fixed objective, starts | coarse candidate set | final claims and plotting |
| Candidate Filter | Bellman candidates and fixed rules | selected/rejected manifests | objective mutation and plotting |
| Warm Start | selected candidate and NLP contract | NLP initial guess | refinement and plotting |
| NLP Refinement | environment, objective, constraints, warm starts | refined response set | Defender optimization and plotting |
| Response Selection | refined response set | Best-found Attacker Response | global-optimum claims and plotting |
| Defender Evaluation | environment and refined response | normalized components and \(J_D\) | use of coarse paths |
| Defender Optimizer | continuous bounds and evaluation callback | placements and Stackelberg solution | plotting |
| Export | typed numerical results and provenance | JSON metadata plus NPZ arrays | modeling and plotting |
| Plotting | exported JSON/NPZ only | figures | numerical computation and optimization |

## 5. Stable data contracts

All public artifacts shall include schema version, run ID, scenario ID, producer, configuration provenance, units, dimensions, and creation metadata. JSON stores metadata, scalars, identifiers, component breakdowns, and array manifests. NPZ stores numerical arrays referenced by those manifests.

### Required artifact families

1. **Environment:** geometry, terrain, sensors, continuous Defender placement, normalization constants, units.
2. **4D stage cost:** axes, shape, valid-state mask, components, combined Attacker stage cost, weight provenance.
3. **2D projection:** source 4D artifact ID, projection rule, projected values, axes, and a visualization-only marker.
4. **Bellman candidates:** candidate/start IDs, switching points, state/control paths, components, feasibility, topology signatures.
5. **Filter manifest:** accepted/rejected IDs, deterministic rules, scores, rejection reasons.
6. **NLP responses:** refined paths/controls, switching points, statuses, residuals, components, parent Bellman IDs.
7. **Best-found response:** selected NLP response ID, refined solution, feasibility evidence, and \(J_A\).
8. **Defender evaluation:** \(z_{sensor}\), refined response ID, normalized PoD, normalized coverage, and \(J_D\).
9. **Stackelberg solution:** final continuous Defender decision, associated refined response, both objective breakdowns, convergence, provenance.

Future phases may add optional fields but shall not rename, reinterpret, or remove required fields without explicit instruction and a schema-version change.

## 6. Objective consistency contract

- Normalization definitions, reference values, clipping policy, and units must be configured and exported.
- Bellman, filtering/ranking, NLP, response selection, and reported Attacker metrics use one Attacker objective contract.
- Every outer-loop evaluation uses one Defender objective contract.
- Diversity may come from state, control, topology, grid, or seeded start generation—not altered objective weights.
- Components are exported separately from weighted totals for auditing.

## 7. Plotting architecture

Plot modules read exported JSON/NPZ artifacts only. They shall not construct geometry, recompute detection, project costs, solve paths, refine trajectories, normalize objectives, or select results.

Required figures:

1. Geometry
2. Projected Cost Map
3. Projected Cost-to-Go Map
4. Projected Cost-to-Go with all Bellman paths and all NLP paths
5. Final Stackelberg Result

Terrain shall have the highest z-order, be filled white, and fully hide heatmaps beneath it. Figure metadata shall identify source artifact IDs.

## 8. Orchestration boundary

The notebook is an orchestration and documentation surface, not a container for monolithic computation. Future executable phases shall import independent modules, pass explicit artifacts, export results before plotting, and keep orchestration cells thin.

Dependency direction:

**configuration → geometry/environment → detection/dynamics/objectives → 4D cost → Bellman/filter/warm start → NLP/selection → Defender evaluation/optimization → export → plotting**

Plotting is a terminal consumer. No computational module may depend on plotting.

## 9. Fixed notebook phase organization

The notebook shall contain the following 15 phases in this order. Each phase has exactly one primary responsibility and may depend only on Configuration or earlier phases. Future-phase references and circular dependencies are forbidden.

| Phase | Name | Single responsibility | Direct dependencies |
|---:|---|---|---|
| 1 | Configuration | Define and validate immutable run inputs, schemas, units, weights, bounds, tolerances, and seeds | none |
| 2 | Terrain Model | Construct terrain representation and terrain-only queries | Phase 1 |
| 3 | Sensor Geometry | Construct sensor position, LOS geometry, masks, and LOS coverage primitives | Phases 1–2 |
| 4 | CasADi Symbolic Detection Model | Define symbolic powered/glide detection quantities and fusion | Phases 1–3 |
| 5 | 4D Stage Cost Construction | Build the authoritative 4D local-cost representation from fixed objectives | Phases 1–4 |
| 6 | 2D Projection | Project exported 4D quantities into visualization-only 2D artifacts | Phases 1 and 5 |
| 7 | Multi-start Bellman | Generate coarse topology, switching-point, path, and warm-start candidates | Phases 1–5 |
| 8 | Bellman Candidate Filtering | Validate, deduplicate, rank, and select the Top-K coarse candidates | Phases 1 and 7 |
| 9 | Bellman to NLP Interface | Transform each selected Bellman candidate into an NLP warm-start contract | Phases 1 and 8 |
| 10 | Attacker CasADi NLP | Continuously refine all supplied Attacker warm starts | Phases 1–5 and 9 |
| 11 | Attacker Best-found Response | Select the best feasible refined response under the unchanged Attacker objective | Phases 1 and 10 |
| 12 | Continuous Defender Optimization | Optimize continuous sensor placement using the complete nested Attacker solver as a black box | Phases 1–11 |
| 13 | Stackelberg Solver | Orchestrate and certify the final nested Defender–Attacker solution record | Phases 1–12 |
| 14 | Export | Serialize authoritative results, metrics, diagnostics, metadata, and arrays | Phases 1–13 |
| 15 | Visualization | Load exported artifacts and render required figures without computation | Phase 14 only |

A phase may use multiple earlier artifacts only when they are necessary for its single stated responsibility. Such use does not authorize that phase to reproduce an earlier responsibility.

## 10. Standard phase interface

Every computational phase shall expose reusable, short, single-task functions. Notebook execution cells in future implementation phases shall call those functions; they shall not contain long algorithm implementations.

Each phase contract shall expose four outputs:

1. **Primary result:** the phase's authoritative typed result.
2. **Validation metrics:** deterministic checks of correctness, feasibility, shape, bounds, and consistency applicable to that phase.
3. **Metadata:** schema version, run/scenario IDs, units, configuration identity, producer, dependencies, seed information, and status.
4. **Export bundle:** JSON-serializable metadata/scalars plus an NPZ array manifest suitable for Phase 14.

Every function shall have explicit inputs and outputs, perform one task, avoid hidden side effects, and avoid mutating global variables. Invalid inputs shall be detected before computation. Failures shall return or raise an explicit failure status with informative diagnostics; silent fallback and silent partial success are forbidden.

## 11. Data-flow, validation, and reproducibility contract

Within every computational phase, the logical flow is fixed:

**Input → Computation → Validation → Export bundle**

Geometry, numerical computation, optimization, validation, export, and visualization remain independent responsibilities. Validation consumes a completed primary result and reports metrics; it shall not silently repair or replace that result. Phase 14 alone writes the authoritative export set. Phase 15 reads that set and never triggers geometry construction, cost-map construction, Bellman, NLP, or Defender optimization.

Given identical explicit inputs, dependency artifacts, configuration, software contract, and seeds, every phase shall produce identical outputs. Every random or multi-start initializer must accept a fixed seed and export it. Output identity shall not depend on notebook execution history or undeclared global state.

Interfaces shall admit future multiplicities and model substitutions—including multiple sensors, multiple attackers, moving sensors, moving targets, higher-order dynamics, and additional sensor models—through typed collections and replaceable module contracts, without reordering or structurally redesigning the 15 phases.

## 12. Future-phase acceptance gates

- Public inputs and outputs conform to stable schemas.
- Continuous variables remain continuous whenever the solver permits; coarse discretization does not redefine the decision.
- Bellman and NLP objectives and normalization constants are identical.
- Every NLP start traces to a filtered Bellman candidate.
- The selected Attacker result is feasible and labeled Best-found Attacker Response.
- Every Defender evaluation references that refined response, never a coarse solution.
- The 2D projection is consumed only for export, reporting, and plotting.
- Numerical artifacts are exported before plotting.
- Geometry does not optimize; optimization does not plot; plotting does not compute.
- Terrain rendering satisfies the white-fill, highest-z-order masking rule.
- Multi-start procedures record seeds and full provenance.

## 13. Non-negotiable invariants

Future phases shall reuse and extend this architecture. They shall not silently change notebook architecture, public interfaces, prior module outputs, Stackelberg hierarchy, Bellman-to-NLP nesting, objective definitions, solver roles, or computation/plot separation.

Any explicitly authorized contract change must document motivation, compatibility impact, schema migration, and validation evidence. Until then, this Phase 0–3 notebook is the governing source of truth.